# MOT17 Pipeline Configurations

This notebook runs the detection -> tracking -> evaluation pipeline for the different
**detector + tracker configurations** supported by this project, and collects the metrics
into a single comparison table.

```text
MOT17 images -> detect.py -> detections CSV -> track.py -> MOT track file -> evaluate.py -> metrics
```

It does **not** re-implement any logic. Each cell simply calls the existing scripts in
`mot_pipeline/` (`detect.py`, `track.py`, `evaluate.py`)
the same way the README does on the command line.

**Before running:** install the dependencies and place the `MOT17` dataset as described in
the README, then run this notebook from the **repo root** (the folder that contains both
this notebook and the `mot_pipeline` folder).

## 1. Configurations

A *configuration* is one detector paired with one tracker. The lists below define which
ones to run. With 3 detectors and 2 trackers there are 6 possible configurations.

On CPU, YOLO26n is fast but Faster R-CNN and DETR are slow, so this defaults to a short
test (`MAX_FRAMES = 30`). Edit the lists and `MAX_FRAMES` to run more.

In [ ]:
import sys
import subprocess
from pathlib import Path

# --- edit these to choose what to run ---
DETECTORS = ["yolo26n", "fasterrcnn", "detr"]   # any of: yolo26n, fasterrcnn, detr
TRACKERS  = ["bytetrack", "ocsort"]              # any of: bytetrack, ocsort
SEQUENCE  = "MOT17-02-DPM"                        # any folder under MOT17/train
MAX_FRAMES = 30                                   # set to None for the whole video
# ----------------------------------------

# This notebook lives in the repo root. The scripts live in the pipeline folder and
# must be run from there (relative imports + the ../MOT17 default path), so point at it.
PIPELINE_DIR = Path.cwd() / "mot_pipeline"
OUTPUT_ROOT = PIPELINE_DIR / "outputs"
PY = sys.executable                # the Python running this notebook

assert PIPELINE_DIR.is_dir(), f"Run this notebook from the repo root; not found: {PIPELINE_DIR}"
configs = [(d, t) for d in DETECTORS for t in TRACKERS]
print("Sequence:", SEQUENCE, "| max frames:", MAX_FRAMES)
print("Configurations to run:")
for d, t in configs:
    print(f"  - {d} + {t}")

In [ ]:
def run(script, extra_args):
    """Call one pipeline script and stream its output, exactly like the command line."""
    cmd = [PY, script, "--sequence", SEQUENCE]
    if MAX_FRAMES is not None:
        cmd += ["--max-frames", str(MAX_FRAMES)]
    cmd += extra_args
    print(">", " ".join(cmd))
    subprocess.run(cmd, cwd=PIPELINE_DIR, check=True)

## 2. Detect

Each detector is run once over the sequence; both trackers later reuse the same boxes.

In [ ]:
for detector in DETECTORS:
    run("detect.py", ["--detector", detector])

## 3. Track

Run every detector + tracker configuration on the saved detections.

In [ ]:
for detector, tracker in configs:
    detections = OUTPUT_ROOT / "detections" / SEQUENCE / f"{detector}.csv"
    run("track.py", ["--tracker", tracker, "--detections", str(detections)])

## 4. Evaluate

Score each configuration's track file with the official TrackEval metrics.

In [ ]:
for detector, tracker in configs:
    tracks = OUTPUT_ROOT / "tracks" / SEQUENCE / f"{detector}_{tracker}.txt"
    run("evaluate.py", ["--tracks", str(tracks)])

## 5. Compare the configurations

`evaluate.py` saved a one-row CSV per configuration. Load them all into one table.

In [ ]:
import pandas as pd

rows = []
for detector, tracker in configs:
    csv_path = OUTPUT_ROOT / "metrics" / SEQUENCE / f"{detector}_{tracker}.csv"
    row = pd.read_csv(csv_path).iloc[0].to_dict()
    row = {"detector": detector, "tracker": tracker, **row}
    rows.append(row)

summary = pd.DataFrame(rows)
# show the headline metrics first; the full set is still in the DataFrame
headline = ["detector", "tracker", "HOTA", "MOTA", "IDF1", "DetA", "AssA"]
summary[headline].round(2)

In [ ]:
# full metrics for every configuration
summary.round(3)